In [1]:
import os
import pandas as pd

# List files in the current working directory
print("Current directory:", os.getcwd())
print("\nAvailable files:")

for file in os.listdir():
    print(file)



df = pd.read_csv("../data/Sample - Superstore.csv", encoding="cp1252")

print("Dataset loaded successfully")
print("Shape:", df.shape)

display(df.head())

Current directory: d:\consulting\Retail Profitability Turnaround\notebook

Available files:
01_load_data.ipynb
02_analyis.ipynb
03_Root-Cause Analysis of Profitability Gaps.ipynb
04_Business_Opportunity_Prioritization.ipynb
04_Product_Region_Profitability.ipynb
05_Discount_Product_Analysis.ipynb
Dataset loaded successfully
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [2]:
# Focus on the two identified problem subcategories
target = df[
    df["Sub-Category"].isin(["Tables", "Bookcases"])
].copy()

# 1. Overall observed profitability
print("Total sales:", round(target["Sales"].sum(), 2))
print("Net profit:", round(target["Profit"].sum(), 2))
print(
    "Negative-profit records:",
    int((target["Profit"] < 0).sum())
)

# 2. Profitability by subcategory and region
region_priority = (
    target.groupby(["Sub-Category", "Region"])
    .agg(
        Sales=("Sales", "sum"),
        Net_Profit=("Profit", "sum"),
        Records=("Profit", "size"),
        Loss_Making_Records=(
            "Profit", lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

region_priority["Profit_Margin_%"] = (
    region_priority["Net_Profit"]
    / region_priority["Sales"] * 100
)

region_priority["Loss_Record_Share_%"] = (
    region_priority["Loss_Making_Records"]
    / region_priority["Records"] * 100
)

display(
    region_priority.sort_values("Net_Profit").round(2)
)

# 3. Rank individual products by net profit
product_priority = (
    target.groupby(["Sub-Category", "Product Name"])
    .agg(
        Sales=("Sales", "sum"),
        Net_Profit=("Profit", "sum"),
        Records=("Profit", "size")
    )
    .reset_index()
)

print(
    "Sum of negative net profits across loss-making products:",
    round(
        product_priority.loc[
            product_priority["Net_Profit"] < 0,
            "Net_Profit"
        ].sum(),
        2
    )
)

display(
    product_priority
    .sort_values("Net_Profit")
    .head(10)
    .round(2)
)

Total sales: 321845.53
Net profit: -21198.04
Negative-profit records: 312


,Sub-Category,Region,Sales,Net_Profit,Records,Loss_Making_Records,Profit_Margin_%,Loss_Record_Share_%
5,Tables,East,39139.81,-11025.38,80,78,-28.17,97.50
6,Tables,South,43916.19,-4623.06,51,27,-10.53,52.94
4,Tables,Central,39154.97,-3559.65,72,49,-9.09,68.06
0,Bookcases,Central,24157.18,-1997.90,50,36,-8.27,72.00
3,Bookcases,West,36004.12,-1646.51,80,32,-4.57,40.00
1,Bookcases,East,43819.33,-1167.63,70,33,-2.66,47.14
2,Bookcases,South,10899.36,1339.49,28,8,12.29,28.57
7,Tables,West,84754.56,1482.61,116,49,1.75,42.24


Sum of negative net profits across loss-making products: -28525.18


,Sub-Category,Product Name,Sales,Net_Profit,Records
78,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,5
72,Tables,Bush Advantage Collection Racetrack Conference...,9544.72,-1934.40,7
53,Tables,Balt Solid Wood Round Tables,6518.75,-1201.06,4
67,Tables,BoxOffice By Design Rectangular and Half-Moon ...,1706.25,-1148.44,3
101,Tables,"Riverside Furniture Oval Coffee Table, Oval En...",4446.18,-1147.40,5
25,Bookcases,O'Sullivan 4-Shelf Bookcase in Odessa Pine,2740.20,-975.10,7
71,Tables,Bretford “Just In Time” Height-Adjustable Mult...,5634.90,-964.19,4
60,Tables,"Bevis Oval Conference Table, Walnut",6942.07,-856.01,8
51,Tables,BPI Conference Tables,2241.87,-795.97,5
34,Bookcases,O'Sullivan Plantations 2-Door Library in Landv...,2946.37,-767.74,5


In [3]:
print("Total sales:", round(target["Sales"].sum(), 2))
print("Net profit:", round(target["Profit"].sum(), 2))
print(
    "Loss-making records:",
    int((target["Profit"] < 0).sum())
)
print(
    "Sum of negative product profits:",
    round(
        product_priority.loc[
            product_priority["Net_Profit"] < 0,
            "Net_Profit"
        ].sum(),
        2
    )
)

Total sales: 321845.53
Net profit: -21198.04
Loss-making records: 312
Sum of negative product profits: -28525.18


In [4]:
display(
    region_priority.sort_values("Net_Profit").round(2)
)

,Sub-Category,Region,Sales,Net_Profit,Records,Loss_Making_Records,Profit_Margin_%,Loss_Record_Share_%
5,Tables,East,39139.81,-11025.38,80,78,-28.17,97.50
6,Tables,South,43916.19,-4623.06,51,27,-10.53,52.94
4,Tables,Central,39154.97,-3559.65,72,49,-9.09,68.06
0,Bookcases,Central,24157.18,-1997.90,50,36,-8.27,72.00
3,Bookcases,West,36004.12,-1646.51,80,32,-4.57,40.00
1,Bookcases,East,43819.33,-1167.63,70,33,-2.66,47.14
2,Bookcases,South,10899.36,1339.49,28,8,12.29,28.57
7,Tables,West,84754.56,1482.61,116,49,1.75,42.24


###  Quantifying and Prioritizing the Business Opportunity

**Objective:** Quantify the profitability problem in Tables and Bookcases and prioritize the products and regions requiring further investigation.

**Key findings**

1. **Overall profitability:** Tables and Bookcases generated $321,845.53 in sales but recorded a net loss of $21,198.04 across 312 loss-making records.

2. **Regional concentration:** Tables in the East were the most severe problem area, recording a loss of $11,025.38, a profit margin of −28.17%, and a loss-making record share of 97.50%. Tables in the South and Central regions recorded further losses of $4,623.06 and $3,559.65, respectively.

3. **Bookcases performance:** Central recorded the largest regional loss for Bookcases at $1,997.90, with a margin of −8.27%. The South was profitable, generating $1,339.49 with a margin of 12.29%.

4. **Product-level concentration:** The five worst-performing products accounted for approximately 29.1% of negative profit summed across all loss-making products. The remaining losses were distributed across other products.

**Business implication:** Prioritize investigating discounting and product economics for Tables in the East, followed by Tables in the South and Central regions and Bookcases in Central. Profitable regional segments provide useful comparison cases for further investigation.

**Interpretation caveat:** These figures quantify observed losses, not recoverable savings. The amount of profit that could be recovered depends on the effects of changes in pricing, demand, costs, and product mix.

**Next step:** Validate and consolidate the findings, check that the conclusions are consistent across analyses, and document the limitations before developing the final recommendations.